# 📥 Step 1: Download and Explore LFW Dataset

This notebook downloads the **Labeled Faces in the Wild (LFW)** dataset and explores its structure.

## Dataset Info
- **LFW**: 13,000+ face images of 5,700+ people
- **Source**: University of Massachusetts
- **Use Case**: Face verification benchmark

In [ ]:
# Imports
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people, fetch_lfw_pairs

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_DIR = PROJECT_ROOT / "data" / "lfw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1.1 Download LFW Dataset

We'll filter to only include people with at least 20 images for better training.

In [ ]:
# Download LFW - this may take a few minutes the first time
MIN_FACES_PER_PERSON = 20

print(f"Downloading LFW dataset (min {MIN_FACES_PER_PERSON} faces per person)...")
print("This may take a few minutes on first run...\n")

lfw_people = fetch_lfw_people(
    min_faces_per_person=MIN_FACES_PER_PERSON,
    resize=1.0,  # Keep original size
    color=True,
    download_if_missing=True
)

print(f"✅ Download complete!")
print(f"   Total images: {len(lfw_people.images)}")
print(f"   Total people: {len(lfw_people.target_names)}")
print(f"   Image shape: {lfw_people.images[0].shape}")

## 1.2 Explore Dataset Structure

In [ ]:
# Dataset structure
images = lfw_people.images
labels = lfw_people.target
names = lfw_people.target_names

print("Dataset Structure:")
print(f"  Images shape: {images.shape}")
print(f"  Labels shape: {labels.shape}")
print(f"  Number of unique people: {len(names)}")
print(f"  Image dtype: {images.dtype}")
print(f"  Pixel range: [{images.min():.2f}, {images.max():.2f}]")

In [ ]:
# Distribution of images per person
unique, counts = np.unique(labels, return_counts=True)
sorted_idx = np.argsort(counts)[::-1]

print("\nTop 15 people by image count:")
print("-" * 40)
for i in sorted_idx[:15]:
    print(f"  {names[unique[i]]}: {counts[i]} images")

print(f"\nMean images per person: {counts.mean():.1f}")
print(f"Median images per person: {np.median(counts):.1f}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(images))
    img = images[idx]
    
    # Normalize for display
    if img.max() <= 1.0:
        img_display = img
    else:
        img_display = img / 255.0
    
    ax.imshow(img_display)
    ax.set_title(names[labels[idx]][:20], fontsize=9)
    ax.axis('off')

plt.suptitle('Sample LFW Images', fontsize=14)
plt.tight_layout()
plt.show()

## 1.3 Download Verification Pairs

LFW provides 6,000 pairs for testing face verification (3,000 same + 3,000 different).

In [ ]:
# Download verification pairs
print("Downloading LFW verification pairs...")

lfw_pairs = fetch_lfw_pairs(subset='test', color=True)

print(f"\n✅ Downloaded {len(lfw_pairs.pairs)} pairs")
print(f"   Same person pairs: {sum(lfw_pairs.target)}")
print(f"   Different person pairs: {len(lfw_pairs.target) - sum(lfw_pairs.target)}")
print(f"   Pair shape: {lfw_pairs.pairs[0].shape}")

In [ ]:
# Visualize pairs
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Same person pairs (top row)
same_indices = np.where(lfw_pairs.target == 1)[0]
for i in range(4):
    idx = same_indices[i]
    pair = lfw_pairs.pairs[idx]
    
    # Combine pair images side by side
    combined = np.hstack([pair[0], pair[1]])
    if combined.max() > 1:
        combined = combined / 255.0
    
    axes[0, i].imshow(combined)
    axes[0, i].set_title('Same Person ✓', color='green')
    axes[0, i].axis('off')

# Different person pairs (bottom row)
diff_indices = np.where(lfw_pairs.target == 0)[0]
for i in range(4):
    idx = diff_indices[i]
    pair = lfw_pairs.pairs[idx]
    
    combined = np.hstack([pair[0], pair[1]])
    if combined.max() > 1:
        combined = combined / 255.0
    
    axes[1, i].imshow(combined)
    axes[1, i].set_title('Different Person ✗', color='red')
    axes[1, i].axis('off')

plt.suptitle('LFW Verification Pairs', fontsize=14)
plt.tight_layout()
plt.show()

## 1.4 Save Raw Data

Save the downloaded data for use in the next notebook.

In [ ]:
# Save raw data
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

np.save(RAW_DIR / "images.npy", images)
np.save(RAW_DIR / "labels.npy", labels)
np.save(RAW_DIR / "names.npy", names)
np.save(RAW_DIR / "pairs.npy", lfw_pairs.pairs)
np.save(RAW_DIR / "pairs_labels.npy", lfw_pairs.target)

print(f"✅ Raw data saved to: {RAW_DIR}")
print(f"\nSaved files:")
for f in RAW_DIR.glob("*.npy"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

## ✅ Step 1 Complete!

**Summary:**
- Downloaded LFW dataset with {len(images)} images of {len(names)} people
- Downloaded {len(lfw_pairs.pairs)} verification pairs for evaluation
- Saved raw data to `data/lfw/raw/`

**👉 Next Step:** Run `02_data_preprocessing.ipynb` to prepare data for training